In [ ]:
import boto3
s3_client = boto3.client('s3')

In [ ]:
type(s3_client)

In [ ]:
s3_client.list_buckets()

In [ ]:
buckets = s3_client.list_buckets()['Buckets']

In [ ]:
bucket_names = [bucket['Name'] for bucket in buckets]

In [ ]:
bucket_names

```bash
#### configure support profile
aws configure --profile itvsupport1

#### edit aws configs
nano ~/.aws/config

#### try to remove folder and its contents
aws s3 rm s3://dg-retail/retail-db/categories --recursive --profile itvsupport1

#### try copy recursively with exclude options
aws s3 cp ~/itversity/Research/data/retail_db/ s3://dg-retail/retail_db/ \
--profile itvsupport1 \
--recursive \
--exclude "*.sql" \
--exclude "README.md" \
--exclude ".git/*"
```

#### Create custom policy: ITVSupportS3RetailDBII 
```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:*"
            ],
            "Resource": "arn:aws:s3:::dg-retail/retail_db*"
        },
        {
            "Effect": "Allow",
            "Action": [
                "s3:List*"
            ],
            "Resource": "*"
        }
    ]
}
```

### aws cli demo: iam
```bash
aws iam list-users --profile itvadmin
aws iam list-groups --profile itvadmin
aws iam list-roles --profile itvadmin
 
#### Lists all AWS Managed Policies as well as custom policies:
aws iam list-policies --profile itvadmin
 
#### List only custom policies:
aws iam list-policies --scope Local --profile itvadmin

#### Create user, add to group, remove from group, and delete user:
aws iam create-user --user-name itvsupport2 --profile itvadmin

aws iam list-users --profile itvadmin

aws iam add-user-to-group \
 --group-name itvsupport \
 --user-name itvsupport2 \
 --profile itvadmin
 
aws iam list-groups-for-user \
 --user-name itvsupport2 \
 --profile itvadmin
```

### aws cli demo: ec2
```bash
#### Describe instances and get the instance ids.
aws ec2 describe-instances \
 --profile itvadmin \
 --region us-west-1
 
aws ec2 describe-instances \
 --profile itvadmin \
 --region us-west-1 | \
grep -i instanceid
 
#### You can use one of the instance ids and get instance status
aws ec2 describe-instance-status \
--instance-id i-07c085b765f162233 \
 --profile itvadmin \
 --region us-west-1

#### Stop the instance and validate whether the instance is stopped or not.
aws ec2 stop-instances \
 --instance-id i-07c085b765f162233 \
 --profile itvadmin \
 --region us-west-1
 
aws ec2 describe-instance-status \
 --instance-id i-07c085b765f162233 \
 --profile itvadmin \
 --region us-west-1

#### Start the instance and validate whether the instance is started or not.
aws ec2 start-instances \
 --instance-id i-07c085b765f162233 i-00f80143dc2e77b85 \
 --profile itvadmin \
 --region us-west-1
 
aws ec2 describe-instance-status \
 --instance-id i-07c085b765f162233 i-00f80143dc2e77b85 \
 --profile itvadmin \
 --region us-west-1
List Elastic IPs that are allocated so far.

aws ec2 describe-addresses \
 --profile itvadmin \
 --region us-west-1

#### save to a json file the ec2 metadata
aws ec2 describe-instances \
 --profile itvadmin \
 --region us-west-1 > instances.json

#### Get only instance ids of all the instances using "query" (like select in sql)
aws ec2 describe-instances \
--query 'Reservations[*].Instances[*].{Instance:InstanceId,status:State.Name}' \
--profile itvadmin \
--region ap-southeast-2

#### filter using metadata key-value pairs (like where in sql)
aws ec2 describe-instances \
--filter Name=instance-type,Values=t2.micro \
--profile itvadmin \
--region ap-southeast-2

#### Get only instance id, type and status of t2.micro instances in stopped state
aws ec2 describe-instances \
--filters Name=instance-type,Values=t2.micro Name=instance-state-name,Values=stopped \
--query 'Reservations[*].Instances[*].{Instance:InstanceId,InstanceType:InstanceType,Status:State.Name}' \
--output json \
--profile itvadmin \
--region us-west-1

#### You can also run these commands to change the instance type using the command line.
aws ec2 stop-instances \
 --instance-id i-07c085b765f162233 \
 --profile itvadmin \
 --region us-west-1
 
aws ec2 modify-instance-attribute \
--instance-id i-07c085b765f162233 \
--instance-type t2.micro \
 --profile itvadmin \
 --region us-west-1
 
aws ec2 describe-instances \
--instance-id i-07c085b765f162233 \
 --profile itvadmin \
 --region us-west-1
 
aws ec2 start-instances \
 --instance-id i-07c085b765f162233 \
 --profile itvadmin \
 --region us-west-1
```

### Exercises:
```bash
#### Create Key Pair or use existing Key Pair.
aws ec2 create-key-pair --key-name createkeypairdemo --region ap-southeast-2
aws ec2 describe-key-pairs --region ap-southeast-2
aws ec2 delete-key-pair --key-name createkeypairdemo --region ap-southeast-2


```

## Using Bootstrapping Scripts
```bash
#### Create instance using aws cli with bootstrap script included
aws ec2 run-instances \
  --image-id ami-013f17f36f8b1fefb \
  --count 1 \
  --instance-type t2.micro \
  --key-name keyname \
  --security-group-ids sg-ID \
  --user-data file://ec2_user_data.sh

#### Creating AMI image from CLI
aws ec2 create-image \
  --instance-id i-ID \
  --name webAppAMI \
  --description "A sample AMI with pre installed apache web server"

#### Use aws cli to create the instance using our AMI
aws ec2 run-instances \
  --image-id ami-ID \
  --count 1 \
  --instance-type t2.micro \
  --key-name keyname \
  --security-group-ids sg-ID
```

### AWS CLI: GLUE
```bash
#### list crawlers
aws glue list-crawlers --profile itvgithub

#### get specific crawler details
aws glue get-crawler --name "Retail Crawler" --profile itvgithub

#### start crawler
aws glue start-crawler --name "Retail Crawler" --profile itvgithub

#### list databases in glue catalog
aws glue get-databases --profile itvgithub

#### list tables in a glue catalog database
aws glue get-tables --database-name retail_db --profile itvgithub

aws glue get-table --database-name retail_db --name products --profile itvgithub
```

## EMR, Hadoop, and HDFS
```bash
### log in to EMR master node using SSH and key pair
### note: might need to open ssh port 22 in the master node security group
ssh -i ~/.ssh/yourkeypair.pem hadoop@ec2-your-master-node-ip.ap-southeast-2.compute.amazonaws.com

### list files in hadoop file system
hadoop fs -ls /

hdfs dfs -ls /

hdfs dfs -ls hdfs:///

### access s3 using hdfs cli
hdfs dfs -ls s3://itv-github/

### download files for 2021-01-17 into master node
mkdir ghactivity && cd ghactivity
wget https://data.gharchive.org/2021-01-17-{0..23}.json.gz

### copy files into s3
hdfs dfs -copyFromLocal * s3://itv-github/landing/ghactivity

### remove files in s3
hdfs dfs -rm s3://itv-github/landing/ghactivity/2021-01-17*

### create s3 bucket for emr itv github project
aws s3 mb s3://itv-github-emr --region ap-southeast-2

### upload files to s3
aws s3 cp . s3://itv-github-emr/prod/landing/ghactivity/ \
    --exclude "*" \
    --include "2021-01-13*" \
    --recursive

### productionizing dev spark code into emr, in the project folder:
### include app.py, process.py, read.py, util.py, write.py only
zip -r itv-ghactivity.zip *.py

### review files in the zip file
unzip -l itv-ghactivity.zip

### copy zip file and main program into emr master node
sudo scp -i ~/.ssh/yoursshkeypair.pem /path/to/zip/file/itv-ghactivity/itv-ghactivity.zip ec2-user@ec2-your-master-node-ip.ap-southeast-2.compute.amazonaws.com:~

sudo scp -i ~/.ssh/yoursshkeypair.pem /path/to/zip/file/itv-ghactivity/app.py ec2-user@ec2-your-master-node-ip.ap-southeast-2.compute.amazonaws.com:~

### login to master node as ec2-user then
mkdir itv-ghactivity
mv itv-ghactivity.zip itv-ghactivity/
mv app.py itv-ghactivity/

### make ec2-user folder in hdfs /user to be able to use spark as ec2-user
sudo -u hdfs hdfs dfs -mkdir /user/ec2-user
### change the ownership to ec2-user
sudo -u hdfs hdfs dfs -chown -R ec2-user:ec2-user /user/ec2-user

### export required env vars by the spark app
export ENVIRON=PROD
export SRC_DIR=s3://itv-github-emr/prod/landing/ghactivity/
export SRC_FILE_FORMAT=json
export TARGET_DIR=s3://itv-github-emr/prod/raw/ghactivity/
export TARGET_FILE_FORMAT=parquet
export SRC_FILE_PATTERN=2021-01-13

### run spark app
spark-submit \
    --master yarn \
    --py-files itv-ghactivity.zip \
    app.py

### verify run by checking target dir, verify in studio cluster nb too
aws s3 ls s3://itv-github-emr/prod/raw/ghactivity/ --recursive --human-readable

### clean up files in raw folder
aws s3 rm s3://itv-github-emr/prod/raw/ghactivity/ --recursive
```
## Deploy Modes for Spark Submit
Apps can be deployed locally, in client (default) which is what we have been using up to now, or the the cluster in the worker nodes. See: [Spark Cluster Overview](https://spark.apache.org/docs/latest/cluster-overview.html)

The env vars are only available in the client instance hence using cluster deploy mode will not work automatically as the worker nodes lack these env vars.

The env vars will need to be passed to the workers during the spark-submit command.

```bash
### run in cluster deploy mode, will fail
spark-submit \
    --master yarn \
    --deploy-mode cluster \
    --py-files itv-ghactivity.zip \
    app.py

### pass env vars to workers in cluster mode, will not work in client mode
spark-submit \
    --master yarn \
    --deploy-mode cluster \
    --py-files itv-ghactivity.zip \
    --conf "spark.yarn.appMasterEnv.ENVIRON=PROD" \
    --conf "spark.yarn.appMasterEnv.SRC_DIR=s3://romadv-itv-github-emr/prod/landing/ghactivity/" \
    --conf "spark.yarn.appMasterEnv.SRC_FILE_FORMAT=json" \
    --conf "spark.yarn.appMasterEnv.TARGET_DIR=s3://romadv-itv-github-emr/prod/raw/ghactivity/" \
    --conf "spark.yarn.appMasterEnv.TARGET_FILE_FORMAT=parquet" \
    --conf "spark.yarn.appMasterEnv.SRC_FILE_PATTERN=2021-01-13" \
    app.py

```
## Step Executions for EMR
Run spark jobs submitted to the EMR Cluster, can be defined in EMR Cluster UI. Can be run in either Client or Cluster (recommended) deployment modes. The main app code and zip file will need to be deployed in s3 so that the worker nodes can access them. When using Client mode, local files in the master node can be referenced such the main app file. The env vars can be added in the Bootstrap stage during Cluster creation by specifying an s3 location for the bootstrap file.

```bash
### copy files spark app files to s3
aws s3 cp itv-ghactivity.zip s3://romadv-itv-github-emr/app/
aws s3 cp app.py s3://romadv-itv-github-emr/app/

### Step execution: spark-submit options
--py-files s3://romadv-itv-github-emr/app/itv-ghactivity.zip
--conf "spark.yarn.appMasterEnv.ENVIRON=PROD"
--conf "spark.yarn.appMasterEnv.SRC_DIR=s3://romadv-itv-github-emr/prod/landing/ghactivity/"
--conf "spark.yarn.appMasterEnv.SRC_FILE_FORMAT=json"
--conf "spark.yarn.appMasterEnv.TARGET_DIR=s3://romadv-itv-github-emr/prod/raw/ghactivity/"
--conf "spark.yarn.appMasterEnv.TARGET_FILE_FORMAT=parquet"
--conf "spark.yarn.appMasterEnv.SRC_FILE_PATTERN=2021-01-14"
```

## EMR Studio 
- Used to run Jupyter notebooks with the EMR cluster attached
- to attach cluster, might need to add AmazonElasticMapReduceEditorsRole policy to the studio role
- see itv-github-nb.ipynb